# A1: Individual Assignment by Harsh Avinash Prakash Kammath

## Introduction

This analysis uses the All Crypto Currencies Dataset from Kaggle, which contains historical price, volume, and market cap data for over 2,000 cryptocurrencies. The dataset includes daily records from as early as 2013, capturing both major assets like Bitcoin (BTC) and thousands of altcoins. The objective of this project is twofold:
1. To build a classification model that predicts the next-day movement of Bitcoin using engineered features, and
2. To apply unsupervised clustering (K-Means) to group cryptocurrencies based on their return-risk profiles. This allows for both predictive and exploratory analysis of digital asset behavior.



## Import all necessary libraries

In this step, we import all required Python libraries for data manipulation, visualization, and modeling.

* pandas and numpy are used for data handling and numerical operations.

* matplotlib, seaborn, and plotly provide visualization capabilities for both static and interactive charts.

* warnings is used to suppress unnecessary warning messages for cleaner outputs.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import seaborn as sns
import plotly.graph_objects as go
import warnings

warnings.filterwarnings("ignore")

## Load the crypto dataset and inspect the structure

We load the historical cryptocurrency market data using pandas.read_csv(). The dataset, sourced from Kaggle, contains daily pricing and market information for a wide range of cryptocurrencies. This step initializes our dataframe (df) for further exploration and analysis.

In [ ]:
# Replace the file path with your file location if needed
df = pd.read_csv('crypto-markets.csv')

## Basic Statistics

We perform basic statistical analysis on the dataset to understand the overall structure, data distribution, and detect any early anomalies. This includes inspecting descriptive metrics such as mean, median, standard deviation, and ranges for key numerical fields like price, volume, and market capitalization.

### Show basic info

In [ ]:
print("Shape of dataset:", df.shape)

Shape of dataset: (942297, 13)


In [ ]:
print("\nColumn names:\n", df.columns.tolist())


Column names:
 ['slug', 'symbol', 'name', 'date', 'ranknow', 'open', 'high', 'low', 'close', 'volume', 'market', 'close_ratio', 'spread']


### Display first 5 rows

In [ ]:
df.head()

,slug,symbol,name,date,ranknow,open,high,low,close,volume,market,close_ratio,spread
0,bitcoin,BTC,Bitcoin,2013-04-28,1,135.30,135.98,132.10,134.21,0.0,1.488567e+09,0.5438,3.88
1,bitcoin,BTC,Bitcoin,2013-04-29,1,134.44,147.49,134.00,144.54,0.0,1.603769e+09,0.7813,13.49
2,bitcoin,BTC,Bitcoin,2013-04-30,1,144.00,146.93,134.05,139.00,0.0,1.542813e+09,0.3843,12.88
3,bitcoin,BTC,Bitcoin,2013-05-01,1,139.00,139.89,107.72,116.99,0.0,1.298955e+09,0.2882,32.17
4,bitcoin,BTC,Bitcoin,2013-05-02,1,116.38,125.60,92.28,105.21,0.0,1.168517e+09,0.3881,33.32


We begin our exploration by understanding the structure of the dataset. With 942,297 rows and 13 columns, the dataset contains historical information on various cryptocurrencies including their open, high, low, and close prices, volume, market capitalization, and spread. We also display the first few rows to validate the presence of key attributes such as symbol, date, and volume, ensuring the data aligns with our expectations.

### Basic Stats of each column

In [ ]:
df.describe()

,ranknow,open,high,low,close,volume,market,close_ratio,spread
count,942297.000000,9.422970e+05,9.422970e+05,9.422970e+05,9.422970e+05,9.422970e+05,9.422970e+05,942297.000000,9.422970e+05
mean,1000.170608,3.483522e+02,4.085930e+02,2.962526e+02,3.461018e+02,8.720383e+06,1.725060e+08,0.459499,1.123400e+02
std,587.575283,1.318436e+04,1.616386e+04,1.092931e+04,1.309822e+04,1.839802e+08,3.575590e+09,0.326160,6.783713e+03
min,1.000000,2.500000e-09,3.200000e-09,2.500000e-10,2.000000e-10,0.000000e+00,0.000000e+00,-1.000000,0.000000e+00
25%,465.000000,2.321000e-03,2.628000e-03,2.044000e-03,2.314000e-03,1.750000e+02,2.958100e+04,0.162900,0.000000e+00
50%,1072.000000,2.398300e-02,2.680200e-02,2.143700e-02,2.389200e-02,4.278000e+03,5.227960e+05,0.432400,0.000000e+00
75%,1484.000000,2.268600e-01,2.508940e-01,2.043910e-01,2.259340e-01,1.190900e+05,6.874647e+06,0.745800,3.000000e-02
max,2072.000000,2.298390e+06,2.926100e+06,2.030590e+06,2.300740e+06,2.384090e+10,3.265025e+11,1.000000,1.770563e+06


To understand the distribution and central tendency of numerical features, we use the .describe() method. This reveals that prices and market-related values span a wide range, with some values (like market and volume) showing high variability, as indicated by large standard deviations. Notably, the presence of zero or near-zero volume and close_ratio values suggests the need for careful filtering before modeling.

## Data Cleaning and preprocessing

This section ensures the dataset is ready for analysis by handling missing values, removing irrelevant or redundant columns, and addressing outliers. Since the raw dataset includes thousands of coins and spans multiple years, we focus only on relevant columns and filter down to active, consistently traded coins to ensure reliability in further statistical and clustering analyses.

### Convert 'date' column to datetime

In [ ]:
df['date'] = pd.to_datetime(df['date'], errors='coerce')

### Drop duplicates

In [ ]:
df = df.drop_duplicates()

### Check for missing values

In [ ]:
missing_summary = df.isnull().sum()
print("Missing values:\n", missing_summary[missing_summary > 0])

Missing values:
 Series([], dtype: int64)


### Sort by date and symbol

In [ ]:
df = df.sort_values(['symbol', 'date']).reset_index(drop=True)

### Preview again

In [ ]:
df.head(5)

,slug,symbol,name,date,ranknow,open,high,low,close,volume,market,close_ratio,spread
0,money,$$$,Money,2015-11-12,1634,0.000013,0.000017,0.000013,0.000017,142.0,0.0,1.0000,0.0
1,money,$$$,Money,2015-11-13,1634,0.000017,0.000033,0.000016,0.000024,85.0,0.0,0.4706,0.0
2,money,$$$,Money,2015-11-14,1634,0.000024,0.000053,0.000023,0.000030,131.0,0.0,0.2333,0.0
3,money,$$$,Money,2015-11-15,1634,0.000030,0.000063,0.000022,0.000035,132.0,0.0,0.3171,0.0
4,money,$$$,Money,2015-11-16,1634,0.000035,0.000046,0.000032,0.000036,280.0,0.0,0.2857,0.0


To ensure data integrity, we first converted the date column to datetime format, allowing for proper chronological sorting. We then removed any duplicate entries to avoid data leakage and checked for missing values to ensure completeness—none were found. Finally, we sorted the dataset by both symbol and date to structure it for time-series analysis and feature engineering.

## Descriptive statistics

To understand the general behavior of each cryptocurrency, we computed descriptive metrics such as average return, volatility, volume, and market capitalization. These statistics offer insight into how assets vary in performance and risk, laying the groundwork for further filtering, analysis, and clustering.

### Summary of numeric columns

In [ ]:
print("Summary statistics:")
display(df.describe())

Summary statistics:


,date,ranknow,open,high,low,close,volume,market,close_ratio,spread
count,942297,942297.000000,9.422970e+05,9.422970e+05,9.422970e+05,9.422970e+05,9.422970e+05,9.422970e+05,942297.000000,9.422970e+05
mean,2017-08-10 04:42:55.641437184,1000.170608,3.483522e+02,4.085930e+02,2.962526e+02,3.461018e+02,8.720383e+06,1.725060e+08,0.459499,1.123400e+02
min,2013-04-28 00:00:00,1.000000,2.500000e-09,3.200000e-09,2.500000e-10,2.000000e-10,0.000000e+00,0.000000e+00,-1.000000,0.000000e+00
25%,2016-12-17 00:00:00,465.000000,2.321000e-03,2.628000e-03,2.044000e-03,2.314000e-03,1.750000e+02,2.958100e+04,0.162900,0.000000e+00
50%,2018-01-18 00:00:00,1072.000000,2.398300e-02,2.680200e-02,2.143700e-02,2.389200e-02,4.278000e+03,5.227960e+05,0.432400,0.000000e+00
75%,2018-07-24 00:00:00,1484.000000,2.268600e-01,2.508940e-01,2.043910e-01,2.259340e-01,1.190900e+05,6.874647e+06,0.745800,3.000000e-02
max,2018-11-30 00:00:00,2072.000000,2.298390e+06,2.926100e+06,2.030590e+06,2.300740e+06,2.384090e+10,3.265025e+11,1.000000,1.770563e+06
std,NaN,587.575283,1.318436e+04,1.616386e+04,1.092931e+04,1.309822e+04,1.839802e+08,3.575590e+09,0.326160,6.783713e+03


To understand the central tendency and dispersion of the dataset, we calculated summary statistics for all numeric columns. This includes the count, mean, standard deviation, minimum, and maximum values, as well as the 25th, 50th (median), and 75th percentiles. These metrics provide a comprehensive overview of the data distribution and help identify potential outliers or inconsistencies in variables such as opening price, closing price, volume, and market capitalization across different cryptocurrencies.

### Number of unique cryptocurrencies

In [ ]:
print("\nNumber of unique cryptocurrencies:", df['symbol'].nunique())


Number of unique cryptocurrencies: 2005


### Date range

In [ ]:
print("Date range:", df['date'].min(), "to", df['date'].max())

Date range: 2013-04-28 00:00:00 to 2018-11-30 00:00:00


### Sample coins

In [ ]:
print("\nSample coins:", df['symbol'].unique()[:10])


Sample coins: ['$$$' '$PAC' '0XBTC' '1337' '1ST' '1WO' '2GIVE' '2GO' '300' '42']


To better understand the scope of the dataset, we examined the number of unique cryptocurrencies tracked. The dataset spans 2005 distinct digital coins recorded between April 28, 2013, and November 30, 2018. This temporal range offers a robust historical perspective on early and mid-stage crypto market activity. A brief look at some of the coin symbols (e.g., ‘$PAC’, ‘0XBTC’, ‘1WO’) reflects the wide diversity of assets covered, including both mainstream and niche tokens.

## Exploratory Data Analysis

In this section, we conduct Exploratory Data Analysis (EDA) to uncover meaningful trends, patterns, and outliers in the cryptocurrency dataset. By visualizing distributions, tracking changes over time, and comparing performance metrics across various coins, EDA enables us to develop preliminary insights and guide further modeling or hypothesis testing. This step is essential to understand the underlying structure and volatility of the crypto market captured in the dataset.

### Pick top 10 coins based on frequency of events (essentially how long the coin has been available)

In [ ]:
coin_counts = df['symbol'].value_counts()
print("Top 10 coins by number of entries:\n", coin_counts.head(10))

Top 10 coins by number of entries:
 symbol
BITS    3189
PXC     2123
BTB     2049
BTM     2043
LTC     2042
NMC     2042
BTC     2042
NVC     2041
PPC     2041
FTC     2037
Name: count, dtype: int64


In [ ]:
# Visualize top 10 coins by frequency
top_symbols = coin_counts.head(10).index.tolist()

In [ ]:
# Filter dataset for only top 10 coins
df_top = df[df['symbol'].isin(top_symbols)].reset_index(drop=True)

# Confirm shape
print("\nFiltered dataset shape:", df_top.shape)


Filtered dataset shape: (21649, 13)


In [ ]:
# Show sample of filtered data
df_top.head()

,slug,symbol,name,date,ranknow,open,high,low,close,volume,market,close_ratio,spread
0,bitstar,BITS,Bitstar,2014-04-23,1460,0.013417,0.013417,0.008505,0.008506,4816.0,32963.0,0.0002,0.0
1,bitstar,BITS,Bitstar,2014-04-24,1460,0.008525,0.008535,0.004697,0.005218,1455.0,22025.0,0.1357,0.0
2,bitstar,BITS,Bitstar,2014-04-25,1460,0.005220,0.005788,0.002506,0.003843,1207.0,17555.0,0.4074,0.0
3,bitstar,BITS,Bitstar,2014-04-26,1460,0.003837,0.004691,0.002199,0.002680,798.0,12795.0,0.1930,0.0
4,bitstar,BITS,Bitstar,2014-04-27,1460,0.002682,0.003577,0.000594,0.000889,423.0,4243.0,0.0989,0.0


The dataset has been filtered to retain only the top 10 cryptocurrencies based on the number of records, indicating coins that have been tracked consistently over time. This ensures the reliability of trends observed in the subsequent analysis. The filtered dataset contains 21,649 rows and 13 columns. A preview of this subset shows relevant features such as date, rank, price metrics (open, high, low, close), volume, and market cap for each coin.

## Data Visualizations

This section explores visual patterns and comparative insights among the top 10 cryptocurrencies. Using time series plots and distribution charts, we examine metrics such as price trends, market capitalization, volume, and volatility. These visualizations are aimed at identifying temporal dynamics, market behavior, and outliers within the selected subset of coins.

### Closing price over time

In [ ]:
fig = px.line(df_top, x='date', y='close', color='symbol',
              title='Closing Price Over Time')
fig.show()

The line chart above visualizes the closing price trends of the top 10 cryptocurrencies over time. It is immediately evident that Bitcoin (BTC) experienced a significant surge in late 2017, peaking close to $20,000 before undergoing a major correction in 2018. In contrast, the majority of the other coins remained clustered near the x-axis, reflecting relatively lower price activity. This visualization highlights the dominance of BTC in terms of valuation and also showcases the volatility that characterizes the cryptocurrency market. The contrast in price movement emphasizes the importance of filtering or rescaling when analyzing coins with vastly different price levels.

### Trading volume over time

In [ ]:
fig = px.line(df_top, x='date', y='volume', color='symbol',
              title='Trading Volume Over Time',
              labels={'volume': 'Daily Volume'})
fig.show()

This line chart displays the daily trading volume for the top 10 cryptocurrencies, measured over time. A sharp surge in trading activity is visible around late 2017 to early 2018, aligning with the cryptocurrency market boom during that period. BTC (Bitcoin) stands out with the highest and most volatile volume spikes, indicating its dominant market influence and higher investor activity. Other coins exhibit relatively stable or muted trading volumes in comparison. This visualization helps highlight liquidity patterns and market attention dynamics across different coins.

### Volatility Over Time (Rolling Std Dev of Returns)

In [ ]:
df_top = df_top.sort_values(['symbol', 'date'])

# Calculate daily return
df_top['daily_return'] = df_top.groupby('symbol')['close'].pct_change()

# 7-day rolling volatility
df_top['rolling_volatility'] = df_top.groupby('symbol')['daily_return'].rolling(7).std().reset_index(level=0, drop=True)

# Plot rolling volatility
fig = px.line(df_top, x='date', y='rolling_volatility', color='symbol',
              title='7-Day Rolling Volatility',
              labels={'rolling_volatility': 'Rolling Std Dev'})
fig.show()

The 7-day rolling standard deviation of daily returns helps capture short-term volatility fluctuations for each cryptocurrency. A clear spike in volatility is observed during late 2017 and early 2018, especially for the dominant symbol (likely BTC), indicating increased market uncertainty during the price surge and subsequent correction. Other cryptocurrencies show lower and more stable volatility throughout the timeline, suggesting varying levels of investor sentiment and trading activity. Overall, this visualization provides critical insight into how price turbulence differed across assets and time.

### Visualizations excluding BTC outlier in first 2 plots and BITS in third plot

In [ ]:
# Replot closing price over time excluding BTC
df_nobtc_price = df_top[df_top['symbol'] != 'BTC']

fig = px.line(df_nobtc_price, x='date', y='close', color='symbol',
              title='Closing Price Over Time (Top Coins Excluding BTC)',
              labels={'close': 'Closing Price', 'date': 'Date'})
fig.show()

Despite Bitcoin’s absence, trading activity shows that certain altcoins experienced surges in volume around late 2017 and early 2018, suggesting broader market interest beyond BTC during the crypto boom. However, volume levels for most coins remain significantly lower compared to the BTC-dominated landscape.

In [ ]:
# Replot trading volume over time excluding BTC
fig = px.line(df_nobtc_price, x='date', y='volume', color='symbol',
              title='Trading Volume Over Time (Top Coins Excluding BTC)',
              labels={'volume': 'Trading Volume', 'date': 'Date'})
fig.show()

Volatility trends indicate sharp fluctuations in early 2018, especially for certain coins like ETH and XRP. The post-2018 period displays relatively reduced and stable volatility, implying market correction and potential maturity among altcoins.

In [ ]:
# Step 9: Remove BITS from df_top for clearer visuals
df_nobits = df_top[df_top['symbol'] != 'BITS']

# Plot rolling volatility without BTC
fig = px.line(df_nobits, x='date', y='rolling_volatility', color='symbol',
              title='7-Day Rolling Volatility (Top 5 Coins, Excluding BTC)',
              labels={'rolling_volatility': 'Rolling Std Dev'})
fig.show()


With BTC removed, the dominance of coins like Litecoin (LTC) and Ethereum (ETH) becomes clearer. These altcoins reached their price peaks during the 2017 bull market but followed with steep corrections, reflecting strong speculative behavior typical in emerging asset classes.

## Data Augmentation (Chose BTC)

BTC was selected as the focal point for augmentation due to its outsized influence on the cryptocurrency market. Its high market capitalization, longest trading history, and significant price and volume volatility make it a representative and insightful choice for generating derived features such as moving averages, volatility bands, and trend indicators. Analyzing BTC in isolation allows deeper insights into market cycles and can enhance modeling accuracy when used as a benchmark.

In [ ]:
# Filter BTC data and create features
df_btc = df_top[df_top['symbol'] == 'BTC'].copy()
df_btc = df_btc.sort_values('date').reset_index(drop=True)

# Create features
df_btc['daily_return'] = df_btc['close'].pct_change()
df_btc['oc_diff'] = df_btc['close'] - df_btc['open']
df_btc['volatility_3d'] = df_btc['daily_return'].rolling(3).std()
df_btc['volatility_7d'] = df_btc['daily_return'].rolling(7).std()
df_btc['rolling_mean_3d'] = df_btc['close'].rolling(3).mean()
df_btc['rolling_mean_7d'] = df_btc['close'].rolling(7).mean()
df_btc['close_lag_1'] = df_btc['close'].shift(1)
df_btc['close_lag_2'] = df_btc['close'].shift(2)

In [ ]:
df_btc.columns

Index(['slug', 'symbol', 'name', 'date', 'ranknow', 'open', 'high', 'low',
       'close', 'volume', 'market', 'close_ratio', 'spread', 'daily_return',
       'rolling_volatility', 'oc_diff', 'volatility_3d', 'volatility_7d',
       'rolling_mean_3d', 'rolling_mean_7d', 'close_lag_1', 'close_lag_2'],
      dtype='object')

In [ ]:
# Step 16: Drop only 'slug', 'symbol', and 'name'

btc_features_df = df_btc.drop(columns=['slug', 'symbol', 'name']).dropna().reset_index(drop=True)

# Preview updated DataFrame
btc_features_df.head()

,date,ranknow,open,high,low,close,volume,market,close_ratio,spread,daily_return,rolling_volatility,oc_diff,volatility_3d,volatility_7d,rolling_mean_3d,rolling_mean_7d,close_lag_1,close_lag_2
0,2013-05-05,1,112.90,118.80,107.14,115.91,0.0,1.288693e+09,0.7521,11.66,0.030311,0.107695,3.01,0.111041,0.107695,108.720000,118.842857,112.50,97.75
1,2013-05-06,1,115.98,124.66,106.64,112.30,0.0,1.249023e+09,0.3141,18.02,-0.031145,0.099637,-3.68,0.092607,0.099637,113.570000,114.237143,115.91,112.50
2,2013-05-07,1,112.25,113.44,97.70,111.50,0.0,1.240594e+09,0.8767,15.74,-0.007124,0.099961,-0.75,0.030971,0.099961,113.236667,110.308571,112.30,115.91
3,2013-05-08,1,109.60,115.78,109.60,113.57,0.0,1.264049e+09,0.6424,6.18,0.018565,0.081859,3.97,0.024860,0.081859,112.456667,109.820000,111.50,112.30
4,2013-05-09,1,113.20,113.46,109.26,112.67,0.0,1.254535e+09,0.8119,4.20,-0.007925,0.069723,-0.53,0.015068,0.069723,112.580000,110.885714,113.57,111.50


To conduct a more focused analysis, we isolated the data for Bitcoin (BTC) and engineered several time-series features that could capture short-term patterns and volatility behavior. Specifically, we calculated percentage change returns (daily_return), opening-closing price difference (oc_diff), short- and medium-term volatility using rolling standard deviation windows (volatility_3d, volatility_7d), and rolling means of closing prices (rolling_mean_3d, rolling_mean_7d). Additionally, lag features (close_lag_1, close_lag_2) were added to account for temporal dependencies. These engineered features were then used to build a new DataFrame, btc_features_df, after dropping irrelevant identifiers and handling missing values. This prepared dataset will serve as the basis for any forecasting or modeling tasks involving BTC.

## Additional Data Visualizations

To further enrich the exploratory analysis, we created supplementary visualizations aimed at uncovering deeper temporal or structural trends in the BTC dataset. These included plots for feature comparisons like daily_return, rolling_mean_3d, and volatility_7d, enabling a visual understanding of how different statistical properties evolve over time. These visual aids support the interpretability of the engineered features and will be useful for evaluating their predictive power in future modeling tasks.

### Correlation Heatmap

In [ ]:
# Correlation heatmap of features
corr=btc_features_df.corr()
threshold=0.5
filtered_corr=corr[corr.abs()>threshold]
fig1=px.imshow(filtered_corr, text_auto='.3f')
fig1.show()

This heatmap provides an intuitive visual representation of the pairwise Pearson correlation coefficients among all numerical features in the btc_features_df dataset. A threshold of 0.5 was applied to filter out weaker correlations, highlighting only moderately to strongly correlated pairs.

Notably, the variables open, high, low, and close show very strong positive correlations with each other, often exceeding 0.99, which is expected in high-frequency financial time series. Additionally, features such as rolling_mean_3d, rolling_mean_7d, and close_lag_1/2 also correlate strongly with price-based columns, confirming the temporal continuity of the closing price trend.

This analysis helps in identifying multicollinearity risks and serves as a valuable guide in feature selection or dimensionality reduction in future modeling steps.

### Pairplot

In [ ]:
# Pairplot-style scatter matrix

fig = px.scatter_matrix(
    btc_features_df,
    dimensions=btc_features_df.columns.tolist(),
    title='Scatter Matrix of BTC Engineered Features',
    height=2400
)
fig.update_traces(diagonal_visible=False)
fig.show()


The pairplot-style scatter matrix provided a granular view of bivariate relationships. It highlighted clear linear clusters among price indicators, market capitalization, and lagged returns. The matrix also emphasized noise in features like daily_return and oc_diff, suggesting their more stochastic behavior compared to smoothed or lagged variables. Overall, this comprehensive visual audit affirms the quality and interdependence of engineered features, setting a strong foundation for subsequent predictive modeling and feature selection efforts.

### Candlestick diagram

In [ ]:
# Candlestick chart of BTC price action

fig = go.Figure(data=[go.Candlestick(
    x=df_btc['date'],
    open=df_btc['open'],
    high=df_btc['high'],
    low=df_btc['low'],
    close=df_btc['close']
)])

fig.update_layout(title='BTC Candlestick Chart',
                  xaxis_title='Date',
                  yaxis_title='Price (USD)')
fig.show()

To understand the historical price fluctuations of Bitcoin, a candlestick chart was generated. This chart provides a clear visual summary of the open, high, low, and close prices for each date in the dataset. It captures major peaks, such as the 2017 surge where BTC crossed $20,000, and subsequent downtrends. By visualizing this with interactive candlesticks, we can better assess market volatility and significant trend reversals over time.

## ML Models

To uncover predictive patterns from the engineered features of the Bitcoin dataset, a suite of supervised machine learning models was implemented. The objective was to forecast relevant target variables—such as future price movements or classification of market states—based on historical and technical indicators. This section evaluates the performance of various algorithms, including regression and classification models, comparing their effectiveness using standard metrics.

### Target Variable Definition

In [ ]:
# Add target column to btc_features_df
btc_features_df['target'] = (btc_features_df['close'].shift(-1) > btc_features_df['close']).astype(int)

# Drop last row (target will be NaN)
btc_features_df = btc_features_df.dropna().reset_index(drop=True)

# Preview target distribution
btc_features_df['target'].value_counts()


,count
target,
1,1105
0,930


To transform the Bitcoin price prediction problem into a supervised classification task, a binary target variable was constructed. This variable captures the directional movement of the closing price: a value of 1 indicates that the next day's closing price is higher than the current day’s, while 0 indicates otherwise. This forward-shifted logic effectively turns the model’s objective into predicting whether the price will go up or down on the next day. After removing the final row (which lacks a future closing price), the resulting class distribution remained relatively balanced, with 1105 instances of upward movement and 930 instances of downward or stagnant movement. This setup ensures meaningful training without extreme class imbalance.

### Dropping highly correlated features

In [ ]:
btc_trimmed_df = btc_features_df[[
    'close', 'volume', 'market', 'spread', 'daily_return',
    'rolling_volatility', 'oc_diff', 'volatility_3d',
    'rolling_mean_3d', 'close_lag_1', 'target'
]]

To reduce multicollinearity and avoid redundant information during model training, highly correlated features were identified and removed from the dataset. This step enhances model interpretability and prevents overfitting. The retained subset includes features that capture distinct aspects of Bitcoin’s behavior, such as price (close), market activity (volume, market), price fluctuation metrics (spread, daily_return, rolling_volatility, volatility_3d, oc_diff), and temporal dynamics (rolling_mean_3d, close_lag_1). The final dataframe, btc_trimmed_df, contains only these selected variables alongside the target label, ensuring that the input features used in model training are both informative and non-redundant.

### Train-test split

In [ ]:
from sklearn.model_selection import train_test_split

# Split features and target
X = btc_trimmed_df.drop(columns='target')
y = btc_trimmed_df['target']

# 80% train, 20% test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

# Confirm shapes
print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

Train shape: (1628, 10)
Test shape: (407, 10)


To prepare the dataset for supervised machine learning, the features (X) and target labels (y) were separated. The target variable represents whether the Bitcoin price increased the following day. An 80-20 split was applied using train_test_split from sklearn.model_selection, ensuring that the class distribution remains consistent across both subsets by applying stratification. The training set contains 1,628 records for model learning, while the test set comprises 407 records used for out-of-sample performance evaluation. This split ensures that model evaluation remains robust and unbiased.

### Training a Decision Tree

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, confusion_matrix

# Initialize and fit model
dt_model = DecisionTreeClassifier(random_state=42)
dt_model.fit(X_train, y_train)

# Predict on test set
y_pred_dt = dt_model.predict(X_test)

# Evaluate
print("Decision Tree Classification Report:")
print(classification_report(y_test, y_pred_dt))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_dt))

Decision Tree Classification Report:
              precision    recall  f1-score   support

           0       0.48      0.56      0.52       186
           1       0.57      0.49      0.53       221

    accuracy                           0.52       407
   macro avg       0.53      0.53      0.52       407
weighted avg       0.53      0.52      0.52       407

Confusion Matrix:
[[104  82]
 [112 109]]


The machine learning section aims to construct and evaluate predictive models to forecast the directional movement of Bitcoin’s closing price. This involves formulating a binary classification task: predicting whether the next day’s closing price will increase (1) or decrease (0) relative to the current day. A new target variable was engineered based on this logic and appended to the feature set. After inspecting the distribution of the target, a balanced split was maintained using stratified train-test sampling (80%-20%) to ensure class representativeness. Highly correlated features were pruned to reduce multicollinearity and prevent overfitting. Subsequently, a Decision Tree classifier was trained and evaluated. The model achieved an accuracy of 52% on the test set, with relatively close precision and recall values across both classes. The confusion matrix suggests moderate predictive capability, though improvements may be necessary through tuning or more sophisticated algorithms (e.g., Random Forest or XGBoost).

### Training a Random Forest Model

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Initialize and fit model
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

# Predict on test set
y_pred_rf = rf_model.predict(X_test)

# Evaluate
print("Random Forest Classification Report:")
print(classification_report(y_test, y_pred_rf))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_rf))

Random Forest Classification Report:
              precision    recall  f1-score   support

           0       0.46      0.39      0.42       186
           1       0.54      0.61      0.57       221

    accuracy                           0.51       407
   macro avg       0.50      0.50      0.50       407
weighted avg       0.50      0.51      0.50       407

Confusion Matrix:
[[ 72 114]
 [ 86 135]]


To improve performance and reduce variance, a Random Forest classifier with 100 estimators was employed. While the model’s overall accuracy was 51%, similar to the Decision Tree, it showed a better F1-score for the positive class (0.57 vs. 0.53), highlighting Random Forest’s strength in capturing complex nonlinear patterns. The confusion matrix reflected improved recall for class 1 but a trade-off in precision and recall for class 0. Overall, the ensemble approach delivered slightly more robust and generalized predictions than the single-tree model.

In [ ]:
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.ensemble import RandomForestClassifier

# Time-aware cross-validator
tscv = TimeSeriesSplit(n_splits=5)

# Define grid
param_grid = {
    'n_estimators': [50, 100],
    'max_depth': [3, 5, 7],
    'min_samples_split': [2, 5]
}

# Setup model
rf = RandomForestClassifier(random_state=42)

# Time-aware GridSearchCV
grid_search = GridSearchCV(estimator=rf, param_grid=param_grid,
                           cv=tscv, scoring='f1', n_jobs=-1)

# Fit
grid_search.fit(X_train, y_train)

# Best model
best_rf = grid_search.best_estimator_
print("Best Params:", grid_search.best_params_)

# Evaluate
from sklearn.metrics import classification_report
y_pred_best = best_rf.predict(X_test)
print(classification_report(y_test, y_pred_best))


Best Params: {'max_depth': 3, 'min_samples_split': 2, 'n_estimators': 100}
              precision    recall  f1-score   support

           0       0.53      0.09      0.15       186
           1       0.55      0.94      0.69       221

    accuracy                           0.55       407
   macro avg       0.54      0.51      0.42       407
weighted avg       0.54      0.55      0.44       407



To further refine model performance, a hyperparameter tuning process was implemented using a GridSearchCV strategy in combination with TimeSeriesSplit to ensure temporal coherence. This time-aware validation scheme is essential when dealing with financial data to avoid forward-looking bias. The grid search explored combinations of key RandomForestClassifier parameters: number of estimators (n_estimators), maximum tree depth (max_depth), and the minimum number of samples required to split an internal node (min_samples_split).

The best combination of parameters identified by the search was:
n_estimators = 100, max_depth = 3, and min_samples_split = 2.

With these optimal parameters, the tuned model yielded an overall accuracy of 55%. Notably, class 1 (indicating price increase) was predicted with a recall of 94%, demonstrating the model’s effectiveness at identifying upward price movements. However, this came at the expense of class 0 (price decline), which suffered from a low recall of 9%, indicating a significant class imbalance in prediction.

This performance reflects a strong bias toward predicting bullish outcomes, a common artifact in financial models where upward momentum is often more persistent. While the tuned model outperformed previous iterations in identifying rises, the imbalance warrants further attention—possibly through re-sampling techniques, cost-sensitive learning, or alternative modeling strategies.

In [ ]:
# Plot feature importances from the best Random Forest model

import plotly.express as px
import pandas as pd

# Get importances and sort
importances = best_rf.feature_importances_
features = X_train.columns
importance_df = pd.DataFrame({'Feature': features, 'Importance': importances})
importance_df = importance_df.sort_values(by='Importance', ascending=False)

# Plot
fig = px.bar(importance_df, x='Importance', y='Feature', orientation='h',
             title='Feature Importances from Tuned Random Forest',
             color='Importance', color_continuous_scale='Viridis')
fig.update_layout(yaxis={'categoryorder':'total ascending'})
fig.show()


To interpret the predictive contribution of each feature, we extracted the feature importances from the best Random Forest model identified through time-aware GridSearchCV. The model was tuned using max_depth = 3, min_samples_split = 2, and n_estimators = 100. Feature importance scores help us understand which features the model relies on most during decision-making.

The top contributing features were market, close, and volume, followed closely by close_lag_1 and rolling_volatility. These features likely capture both the immediate market state and short-term price momentum, which are relevant in forecasting short-term price direction. On the other hand, features like rolling_mean_3d and spread had relatively lower importance, indicating limited influence in the model’s predictions.

Overall, the model leveraged both lagged and volatility-based indicators, confirming the relevance of temporal and market variability measures in classifying Bitcoin market movements.

In [ ]:
#Train RF with class_weight balanced

balanced_rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=3,
    min_samples_split=2,
    class_weight='balanced',
    random_state=42
)

balanced_rf.fit(X_train, y_train)
y_pred_balanced = balanced_rf.predict(X_test)

# Evaluate
print("Random Forest (class_weight=balanced) Report:")
print(classification_report(y_test, y_pred_balanced))
print(confusion_matrix(y_test, y_pred_balanced))


Random Forest (class_weight=balanced) Report:
              precision    recall  f1-score   support

           0       0.47      0.50      0.48       186
           1       0.55      0.52      0.53       221

    accuracy                           0.51       407
   macro avg       0.51      0.51      0.51       407
weighted avg       0.51      0.51      0.51       407

[[ 93  93]
 [107 114]]


To address the potential class imbalance in the dataset, a Random Forest model was trained with the class_weight='balanced' parameter. This instructs the model to automatically adjust weights inversely proportional to class frequencies in the training data, which helps it better handle skewed distributions. The model retained the same hyperparameters identified in the grid search: n_estimators=100, max_depth=3, and min_samples_split=2.

The classification performance with this balanced Random Forest yielded an accuracy of 51%, which remained comparable to earlier models. However, the recall for the minority class (label 1) improved slightly to 0.52, while the precision was 0.55, indicating a better trade-off between correctly identifying the positive class and avoiding false positives. The F1-score for class 1 was 0.53, which was more balanced compared to the unweighted versions.

From the confusion matrix:
[[ 93  93]
 [107 114]]
we see a more even distribution of predictions across classes compared to the previous Random Forest. The model correctly identified 114 true positives and 93 true negatives, showing that class balancing helped reduce bias toward the majority class. However, there's still room for improvement, particularly in reducing the false negative count (107), which could be targeted in future modeling iterations.

In [ ]:
# Combine with correct date source

# Use the index of X_test to align with df_btc
test_index = X_test.index
date_test = df_btc.loc[test_index, 'date']  # from original df_btc that still has 'date'

# Create comparison DataFrame
results_df = pd.DataFrame({
    'date': date_test.values,
    'actual': y_test.values,
    'predicted': y_pred_balanced
}).sort_values('date').reset_index(drop=True)

# Plot using Plotly
import plotly.graph_objects as go

fig = go.Figure()
fig.add_trace(go.Scatter(x=results_df['date'], y=results_df['actual'],
                         mode='lines+markers', name='Actual (BTC Up=1, Down=0)'))
fig.add_trace(go.Scatter(x=results_df['date'], y=results_df['predicted'],
                         mode='lines+markers', name='Predicted'))

fig.update_layout(title='Actual vs. Predicted BTC Movement (Test Set)',
                  xaxis_title='Date', yaxis_title='Class (0 = Down, 1 = Up)',
                  height=500)
fig.show()


To better interpret the model’s predictive behavior on unseen data, a time-series line plot was constructed using Plotly to visualize both actual and predicted Bitcoin movement classes (0 = Down, 1 = Up). This visualization allows us to assess not just classification accuracy, but how prediction errors are distributed over time.

The model’s outputs were mapped to their corresponding dates from the original df_btc DataFrame to ensure temporal alignment. A side-by-side trace of actual (y_test) and predicted (y_pred_balanced) values was created using go.Scatter, and both were plotted as overlapping line+marker traces. This approach emphasizes sudden changes in class values while also preserving the trend structure.

As seen in the plot, the predicted line (red) generally follows the actual trend (blue), with some visible divergence in certain regions. The dense vertical switching indicates the inherent volatility of BTC movement classification and challenges in achieving robust signal capture. Despite achieving a modest test accuracy of 51% (refer Section 1.10), this visual reinforces the model’s ability to track general market direction over time, albeit with occasional misclassifications.

Such visual diagnostic tools are essential in evaluating classification models trained on sequential financial data, where a simple score metric might obscure volatility-related behavior.

In [ ]:
# Add correctness column
results_df['correct'] = (results_df['actual'] == results_df['predicted']).astype(int)

# Plot correct (1) vs incorrect (0) by date
fig = px.scatter(results_df, x='date', y='correct',
                 color='correct',
                 color_discrete_map={1: 'green', 0: 'red'},
                 title='Prediction Accuracy Over Time (1 = Correct, 0 = Incorrect)',
                 labels={'correct': 'Prediction Correct?'})
fig.show()


### Clustering through K Means

To uncover latent structures and patterns within the Bitcoin dataset, unsupervised learning was performed using the K-Means clustering algorithm. Unlike supervised approaches, clustering does not rely on target labels but instead groups data points based on intrinsic similarities across selected features. This technique is particularly useful in financial analytics for identifying hidden regimes, market behaviors, or anomalous patterns that might not be evident from labeled classifications.

The primary objective of applying K-Means here is to investigate whether distinct Bitcoin market behaviors can be separated into meaningful clusters—potentially reflecting bullish, bearish, or volatile trading regimes. By analyzing the resulting clusters and their centroids, we can gain deeper insights into the multidimensional structure of the data, which can later support segmentation or targeted strategy development.

In [ ]:
# Aggregate coin-level features for clustering

df_top = df_top.sort_values(['symbol', 'date'])

# Compute daily return
df_top['daily_return'] = df_top.groupby('symbol')['close'].pct_change()

# Aggregate features per coin
agg_df = df_top.groupby('symbol').agg({
    'daily_return': ['mean', 'std'],
    'volume': 'mean',
    'market': 'mean'
}).reset_index()

# Rename columns
agg_df.columns = ['symbol', 'avg_return', 'volatility', 'avg_volume', 'avg_market_cap']

# Preview
agg_df


,symbol,avg_return,volatility,avg_volume,avg_market_cap
0,BITS,10.466408,20.236627,1.973164e+04,6.713290e+05
1,BTB,0.882618,6.613330,8.236676e+02,2.336899e+05
2,BTC,0.002660,0.044018,1.450143e+09,3.785297e+10
3,BTM,0.540101,1.829253,8.226136e+06,7.404508e+07
4,FTC,0.005339,0.123407,1.908622e+05,8.622189e+06
5,LTC,0.003372,0.073950,1.383520e+08,1.634127e+09
6,NMC,0.003738,0.096547,2.533249e+05,1.509807e+07
7,NVC,0.003111,0.089231,9.350097e+04,3.794229e+06
8,PPC,0.003242,0.079207,4.835195e+05,2.868602e+07
9,PXC,0.202452,0.933598,9.786923e+03,1.289483e+05


To prepare for clustering, the dataset was first aggregated at the coin level to extract stable and interpretable characteristics for each cryptocurrency. This involved computing the daily return using the percentage change in closing prices, followed by calculating the mean return, volatility (standard deviation of daily returns), average trading volume, and average market capitalization for each coin.

The final output, agg_df, represents one row per coin with these derived metrics, forming the core feature space for unsupervised clustering. These features collectively summarize a coin’s performance and market behavior over time, making them suitable for pattern recognition through K-Means.

In [ ]:
# Add Sharpe and Sortino ratios

# Recompute just in case
df_top['daily_return'] = df_top.groupby('symbol')['close'].pct_change()

# Separate positive and negative returns
returns = df_top[['symbol', 'daily_return']].dropna()
returns['negative_return'] = returns['daily_return'].apply(lambda x: x if x < 0 else 0)

# Compute stats
sharpe = returns.groupby('symbol')['daily_return'].mean() / returns.groupby('symbol')['daily_return'].std()
sortino = returns.groupby('symbol')['daily_return'].mean() / returns.groupby('symbol')['negative_return'].std()

# Merge into agg_df
agg_df['sharpe_ratio'] = agg_df['symbol'].map(sharpe)
agg_df['sortino_ratio'] = agg_df['symbol'].map(sortino)

# Preview
agg_df

,symbol,avg_return,volatility,avg_volume,avg_market_cap,sharpe_ratio,sortino_ratio
0,BITS,10.466408,20.236627,1.973164e+04,6.713290e+05,0.517201,23.005071
1,BTB,0.882618,6.613330,8.236676e+02,2.336899e+05,0.133460,5.484334
2,BTC,0.002660,0.044018,1.450143e+09,3.785297e+10,0.060429,0.099960
3,BTM,0.540101,1.829253,8.226136e+06,7.404508e+07,0.295258,1.939418
4,FTC,0.005339,0.123407,1.908622e+05,8.622189e+06,0.043264,0.093073
5,LTC,0.003372,0.073950,1.383520e+08,1.634127e+09,0.045599,0.097799
6,NMC,0.003738,0.096547,2.533249e+05,1.509807e+07,0.038722,0.079124
7,NVC,0.003111,0.089231,9.350097e+04,3.794229e+06,0.034866,0.069598
8,PPC,0.003242,0.079207,4.835195e+05,2.868602e+07,0.040933,0.078972
9,PXC,0.202452,0.933598,9.786923e+03,1.289483e+05,0.216851,0.957445


To evaluate the risk-adjusted performance of the top cryptocurrencies, we computed the Sharpe and Sortino ratios using daily returns. The Sharpe ratio compares average returns to overall volatility, while the Sortino ratio isolates downside risk—focusing only on negative return deviations. As shown in the table, BITS exhibits the highest Sharpe (0.5172) and Sortino (23.01) ratios, indicating a favorable return profile relative to both total and downside risk. In contrast, more established coins like BTC and LTC show relatively low risk-adjusted returns despite their larger market caps. This suggests that smaller-cap assets like BITS may offer higher returns per unit of risk, though they may also come with higher volatility.

In [ ]:
# Normalize features and apply K-Means clustering

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

# Select feature columns
feature_cols = ['avg_return', 'volatility', 'avg_volume',
                'avg_market_cap', 'sharpe_ratio', 'sortino_ratio']

# Normalize
scaler = StandardScaler()
scaled_features = scaler.fit_transform(agg_df[feature_cols])

# Apply K-Means
kmeans = KMeans(n_clusters=3, random_state=42)
clusters = kmeans.fit_predict(scaled_features)

# Add cluster labels
agg_df['cluster'] = clusters

# Preview clustered data
agg_df.sort_values('cluster')

,symbol,avg_return,volatility,avg_volume,avg_market_cap,sharpe_ratio,sortino_ratio,cluster
1,BTB,0.882618,6.613330,8.236676e+02,2.336899e+05,0.133460,5.484334,0
3,BTM,0.540101,1.829253,8.226136e+06,7.404508e+07,0.295258,1.939418,0
5,LTC,0.003372,0.073950,1.383520e+08,1.634127e+09,0.045599,0.097799,0
4,FTC,0.005339,0.123407,1.908622e+05,8.622189e+06,0.043264,0.093073,0
6,NMC,0.003738,0.096547,2.533249e+05,1.509807e+07,0.038722,0.079124,0
7,NVC,0.003111,0.089231,9.350097e+04,3.794229e+06,0.034866,0.069598,0
9,PXC,0.202452,0.933598,9.786923e+03,1.289483e+05,0.216851,0.957445,0
8,PPC,0.003242,0.079207,4.835195e+05,2.868602e+07,0.040933,0.078972,0
2,BTC,0.002660,0.044018,1.450143e+09,3.785297e+10,0.060429,0.099960,1
0,BITS,10.466408,20.236627,1.973164e+04,6.713290e+05,0.517201,23.005071,2


To identify patterns among crypto assets, we applied K-Means clustering to key normalized features: average return, volatility, average volume, market capitalization, Sharpe ratio, and Sortino ratio. Using 3 clusters (with random_state=42 for reproducibility), the algorithm grouped assets based on similar risk-return and market behavior profiles.

In the previewed output, Cluster 0 includes assets like BTB, BTM, LTC, and FTC—characterized by relatively low risk-adjusted returns and moderate to low market activity. Subsequent analysis of all clusters will help distinguish high-performing outliers from lower-performing or more stable coins, aiding in strategic grouping or investment insights.

In [ ]:
# Use top 100 coins by data count

coin_counts = df['symbol'].value_counts()
top_symbols_100 = coin_counts.head(100).index.tolist()  # or use .head(2000) for full set

df_top_extended = df[df['symbol'].isin(top_symbols_100)].copy()
df_top_extended = df_top_extended.sort_values(['symbol', 'date'])

In [ ]:
# Aggregate features for top 100 coins (same metrics as before)

# 1. Get top 100 coins by frequency
coin_counts = df['symbol'].value_counts()
top_symbols_100 = coin_counts.head(100).index.tolist()

# 2. Filter and sort
df_top_100 = df[df['symbol'].isin(top_symbols_100)].copy()
df_top_100 = df_top_100.sort_values(['symbol', 'date'])

# 3. Compute daily return
df_top_100['daily_return'] = df_top_100.groupby('symbol')['close'].pct_change()

# 4. Compute Sharpe and Sortino
returns = df_top_100[['symbol', 'daily_return']].dropna()
returns['neg_return'] = returns['daily_return'].apply(lambda x: x if x < 0 else 0)

sharpe = returns.groupby('symbol')['daily_return'].mean() / returns.groupby('symbol')['daily_return'].std()
sortino = returns.groupby('symbol')['daily_return'].mean() / returns.groupby('symbol')['neg_return'].std()

# 5. Aggregate main features
agg_df_100 = df_top_100.groupby('symbol').agg({
    'daily_return': ['mean', 'std'],
    'volume': 'mean',
    'market': 'mean'
}).reset_index()

agg_df_100.columns = ['symbol', 'avg_return', 'volatility', 'avg_volume', 'avg_market_cap']

# 6. Add sharpe and sortino
agg_df_100['sharpe_ratio'] = agg_df_100['symbol'].map(sharpe)
agg_df_100['sortino_ratio'] = agg_df_100['symbol'].map(sortino)

# Drop any rows with NaNs
agg_df_100 = agg_df_100.dropna().reset_index(drop=True)

# Preview
agg_df_100.head()


,symbol,avg_return,volatility,avg_volume,avg_market_cap,sharpe_ratio,sortino_ratio
0,42,0.040938,1.264915,2625.036786,3.615870e+05,0.032364,0.496006
1,ABY,0.009098,0.136998,61100.316176,2.132168e+06,0.066408,0.150474
2,AC,0.047400,0.545399,3399.270695,1.951298e+06,0.086908,0.408205
3,ANC,0.036721,0.617815,12275.284394,1.100714e+06,0.059437,0.399147
4,ARG,0.043123,1.027802,1269.410326,1.955117e+05,0.041957,0.464015


In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

# Normalize and cluster

# Select features for clustering
feature_cols = ['avg_return', 'volatility', 'avg_volume',
                'avg_market_cap', 'sharpe_ratio', 'sortino_ratio']

# Normalize
scaler = StandardScaler()
scaled_features_100 = scaler.fit_transform(agg_df_100[feature_cols])

# KMeans clustering (e.g., 3 clusters)
kmeans = KMeans(n_clusters=3, random_state=42)
agg_df_100['cluster'] = kmeans.fit_predict(scaled_features_100)

# Preview
agg_df_100.sort_values('cluster')


,symbol,avg_return,volatility,avg_volume,avg_market_cap,sharpe_ratio,sortino_ratio,cluster
0,42,0.040938,1.264915,2625.036786,3.615870e+05,0.032364,0.496006,0
1,ABY,0.009098,0.136998,61100.316176,2.132168e+06,0.066408,0.150474,0
2,AC,0.047400,0.545399,3399.270695,1.951298e+06,0.086908,0.408205,0
3,ANC,0.036721,0.617815,12275.284394,1.100714e+06,0.059437,0.399147,0
4,ARG,0.043123,1.027802,1269.410326,1.955117e+05,0.041957,0.464015,0
...,...,...,...,...,...,...,...,...
33,FAIR,3.840961,10.537801,160041.293225,7.597716e+06,0.364494,11.168502,1
9,BET,5.401549,14.586110,7872.574263,1.944764e+06,0.370321,13.346707,1
39,GCC,6.447010,43.019362,8257.741792,8.679050e+05,0.149863,20.713898,1
73,RED,3.589251,11.858781,79919.418153,3.531829e+05,0.302666,12.189478,1


We first computed daily returns per symbol and derived Sharpe and Sortino ratios by separating positive and negative returns. These risk-adjusted performance metrics were merged into an aggregated dataframe. Then, we filtered the dataset to retain the top 100 coins by data availability and computed average return, volatility, volume, and market cap. After cleaning missing values, we normalized these features using StandardScaler and applied K-Means clustering (k=3) to segment the coins based on their risk-return profiles. Cluster labels were added to enable further analysis of similarly performing asset groups.

In [ ]:
# 2D Scatterplot of clusters

fig = px.scatter(
    agg_df_100,
    x='volatility',
    y='avg_return',
    color='cluster',
    hover_data=['symbol', 'sharpe_ratio', 'sortino_ratio', 'avg_market_cap'],
    title='K-Means Clustering of Top 100 Coins (Return vs Volatility)',
    labels={'volatility': 'Volatility', 'avg_return': 'Avg Return'}
)
fig.update_traces(marker=dict(size=8))
fig.show()


This visualization shows the result of K-Means clustering applied to the top 100 cryptocurrencies, using two key dimensions:

* X-axis: Volatility

* Y-axis: Average Return

* Color: Cluster assignment (0, 1, 2)

The clustering was performed on six normalized features: avg_return, volatility, avg_volume, avg_market_cap, sharpe_ratio, and sortino_ratio. Coins were grouped into 3 clusters using KMeans (n_clusters=3).

Each dot represents one cryptocurrency, and the color encodes its assigned cluster. The goal is to identify distinct groups based on risk (volatility) and performance (return). For example:

* Cluster 0 (dark purple): Low-risk, low-return assets

* Cluster 1 (orange-pink): Higher return but also higher volatility

* Cluster 2 (yellow): Extreme outlier with very high return and volatility

Hovering over a point (in the actual plot) reveals additional metrics like Sharpe Ratio, Sortino Ratio, and Market Cap, helping deepen the interpretation of each cluster.

## Conclusion

In this notebook, we conducted a comprehensive quantitative analysis of the top 100 cryptocurrencies using return-based metrics and clustering techniques. The workflow included the following steps:

* Data preprocessing: Selected the top 100 coins based on data availability, computed daily returns, and calculated key performance indicators such as Sharpe and Sortino ratios.

* Feature engineering: Aggregated statistical features including average return, volatility, volume, market cap, and risk-adjusted returns.

* K-Means clustering: Applied unsupervised learning on scaled features to group similar cryptocurrencies based on their performance and risk profile.

* Visualization: Produced a 2D scatterplot (Return vs Volatility) to interpret cluster groupings and identify distinct asset classes among cryptocurrencies.

Key Insights:

* Cluster 0 represents stable, low-return assets with minimal volatility — likely mature coins with large market caps.

* Cluster 1 includes moderately volatile coins with better risk-adjusted returns — potential candidates for diversified portfolios.

* Cluster 2 contains extreme outliers with very high volatility and returns — possibly speculative or recently surging coins.

These findings can guide investors in identifying assets that align with different risk preferences and investment strategies, while the clustering approach provides a scalable framework for grouping and comparing cryptocurrencies beyond surface-level metrics.

# References

OpenAI. (2024). ChatGPT (July 2024 version) [Large language model]. https://chat.openai.com
→ For analytical guidance, and documentation assistance throughout this notebook.